# MIMIC-IV & MIMIC-CXR Exploratory Analysis

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import config
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 1000)

## Raw data files

In [ ]:
def check_directory(path):
    if not path.exists():
        print(f"{path}: Directory not found")
        return
    files = sorted(path.glob("*"))
    if not files:
        print(f"{path}: Empty directory")
        return
    for f in files:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"{f.name:<35} {size_mb:8.2f} MB")

print("MIMIC-IV files:")
check_directory(config.MIMIC_IV_DIR)
print("\nMIMIC-CXR files:")
check_directory(config.MIMIC_CXR_DIR)

## ICU admissions

In [ ]:
icustays_path = config.MIMIC_IV_DIR / "icustays.csv.gz"
if not icustays_path.exists():
    icustays_path = config.MIMIC_IV_DIR / "icustays.csv"

if icustays_path.exists():
    df_icu = pd.read_csv(icustays_path)
    print(f"Total ICU stays: {len(df_icu)}")
    print(f"Unique patients: {df_icu['subject_id'].nunique()}")
    display(df_icu.head())
    
    # Summary of ICU length of stay
    print("\nLength of stay (days):")
    print(df_icu['los'].describe())
else:
    print(f"File not found: {icustays_path}")

## Item IDs for vitals and labs

In [ ]:
# Vitals item IDs from config
for vital, itemids in config.VITAL_ITEM_IDS.items():
    print(f"{vital:<15}: {itemids}")

print()
# Labs item IDs from config
for lab, itemids in config.LAB_ITEM_IDS.items():
    print(f"{lab:<15}: {itemids}")

## Chartevents schema sample

In [ ]:
# Read small sample to verify column names
chart_path = config.MIMIC_IV_DIR / "chartevents.csv.gz"
if not chart_path.exists():
    chart_path = config.MIMIC_IV_DIR / "chartevents.csv"

if chart_path.exists():
    df_chart = pd.read_csv(chart_path, nrows=100)
    display(df_chart.head())
else:
    print(f"File not found: {chart_path}")

## Chest X-Ray metadata

In [ ]:
cxr_meta_path = config.MIMIC_CXR_DIR / "mimic-cxr-2.0.0-metadata.csv.gz"
if not cxr_meta_path.exists():
    cxr_meta_path = config.MIMIC_CXR_DIR / "mimic-cxr-2.0.0-metadata.csv"

if cxr_meta_path.exists():
    df_cxr = pd.read_csv(cxr_meta_path)
    print(f"Total CXR records: {len(df_cxr)}")
    print(f"Unique patients: {df_cxr['subject_id'].nunique()}")
    display(df_cxr.head())
    
    # View position distribution
    print("\nView positions:")
    print(df_cxr['ViewPosition'].value_counts(dropna=False))
    
    frontal = df_cxr['ViewPosition'].isin(['PA', 'AP']).sum()
    print(f"\nFrontal scans (PA/AP): {frontal} ({frontal / len(df_cxr) * 100:.1f}%)")
else:
    print(f"File not found: {cxr_meta_path}")